# Huggingface Dataset Loader with DataOptim

#### Shikra-RD、GQA、RefCOCO/g/+ Datasets

In [ ]:
from datasets import load_dataset, get_dataset_split_names
from pathlib import Path
import json
# print(get_dataset_split_names("BAAI/DataOptim"))
# 可能是 ["train"] 或 ["train", "validation", "test"]
folder = Path("../DataOptim")
folder.mkdir(parents=True, exist_ok=True)


base = "hf://datasets/BAAI/DataOptim"
ds_mix = load_dataset(
    "json",
    data_files={
        "shikra": f"{base}/data/shikra.json",
        "gqa": f"{base}/data/gqa.json",
        "ref3reg": f"{base}/data/ref3reg.json",
        "ref3rec": f"{base}/data/ref3rec.json",
    },
    repo_id="BAAI/DataOptim",
)

for dataset_name in ds_mix.keys():
    with open(folder / f"{dataset_name}.jsonl", "w") as f:
        for item in ds_mix[dataset_name]:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


In [ ]:
print(ds_mix["shikra"][0])

## Deal with Images

In [19]:
folders = set()
for name in ds_mix.keys():
    for item in ds_mix[name]:
        img_path = item["image"]
        folder_name = img_path.split("/")[0]
        folders.add(folder_name)

print(f"需要下載的圖片資料夾: {folders}")

需要下載的圖片資料夾: {'flickr30k', 'visual_genome', 'coco'}


In [25]:
import requests
from pathlib import Path
import zipfile
from tqdm import tqdm

HF_BASE = "https://huggingface.co/datasets/BAAI/DataOptim/resolve/main/images"

images = folder / "images"      # folder 是你外面定義的 Path
images.mkdir(parents=True, exist_ok=True)

try:
    url = f"{HF_BASE}/coco/train2014.zip"
    zip_path = images / "train2014.zip"

    # 先拿檔案大小（做進度條 total）
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    chunk_size = 8192

    with open(zip_path, "wb") as f, tqdm(
        total=total,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc="Downloading coco/train2014.zip",
    ) as pbar:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

    # 解壓縮到 images/coco/
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(images / "coco")

    # 刪掉 zip 檔
    zip_path.unlink()

    print("coco 下載並解壓完成")

except Exception as e:
    print("下載圖片失敗:", e)


coco 下載並解壓完成


## 確認檔案或路徑沒有錯誤

In [ ]:
json_files = list(folder.glob("*.jsonl"))

for json_file in json_files:
    dataset_name = json_file.stem
    print(f"處理 {dataset_name} 的圖片...")
    with open(json_file, "r") as f:
        for line in f:
            item = json.loads(line)
            img_path = item["image"]
            folder_name = img_path.split("/")[0]
            img_filename = img_path.split("/")[-1]
            src_path = images / folder_name / img_filename
            dest_folder = folder / "images" / folder_name
            dest_folder.mkdir(parents=True, exist_ok=True)
            dest_path = dest_folder / img_filename
            if not dest_path.exists():
                if src_path.exists():
                    dest_path.write_bytes(src_path.read_bytes())
                else:
                    print(f"圖片不存在: {src_path}")
    print(f"{dataset_name} 的圖片處理完成。")

處理 gqa 的圖片...
gqa 的圖片處理完成。
處理 ref3rec 的圖片...
ref3rec 的圖片處理完成。
處理 shikra 的圖片...
shikra 的圖片處理完成。
處理 ref3reg 的圖片...
ref3reg 的圖片處理完成。


In [ ]:
from pathlib import Path
import json

folder = Path("../DataOptim")
json_files = list(folder.glob("*.jsonl"))

for json_file in json_files:
    dataset_name = json_file.stem
    with open(json_file, "r") as f:
        for line in f:
            item = json.loads(line)
            print(item)
            break
    break

{'id': 'gqa_02930152', 'image': 'visual_genome/2354786.jpg', 'conversations': [{'from': 'human', 'value': '<image>\nIs the sky dark? Answer the question directly with a short sentence or phrase.'}, {'from': 'gpt', 'value': 'Yes, the sky is dark.'}]}
